In [20]:
import pandas as pd
import numpy as np

train = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/train.csv", parse_dates=["date"])
test  = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/test.csv",  parse_dates=["date"])

all_df = pd.concat([train.assign(is_train=1), test.assign(is_train=0)], ignore_index=True)
all_df = all_df.sort_values(["store_nbr","family","date"])

In [21]:
all_df["dow"] = all_df["date"].dt.dayofweek  # Mon=0 ... Sun=6

# dummies: base jueves (3)
dow_dummies = pd.get_dummies(all_df["dow"], prefix="dow")
dow_dummies = dow_dummies.drop(columns=["dow_3"]) 
all_df = pd.concat([all_df, dow_dummies], axis=1)

# payday features
all_df["day"] = all_df["date"].dt.day
all_df["is_payday"] = ((all_df["date"].dt.day == 15) | (all_df["date"].dt.is_month_end)).astype(int)

In [22]:
all_df["dom_bucket_5"] = ((all_df["date"].dt.day - 1) // 5).astype(int)

In [23]:
eq_start = pd.Timestamp("2016-04-16")
eq_end   = pd.Timestamp("2016-05-31")
all_df["is_earthquake_window"] = ((all_df["date"] >= eq_start) & (all_df["date"] <= eq_end)).astype(int)

In [24]:
stores = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/stores.csv")
all_df = all_df.merge(stores, on="store_nbr", how="left")

In [33]:
oil = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/oil.csv",
                  parse_dates=["date"])

oil["dcoilwtico"] = oil["dcoilwtico"].ffill()
train_max_date = train["date"].max()
oil_mean = oil.loc[oil["date"] <= train_max_date, "dcoilwtico"].mean()
oil["oil_above_mean"] = (oil["dcoilwtico"] > oil_mean).astype(int)

all_df = all_df.merge(oil[["date","dcoilwtico","oil_above_mean"]], on="date", how="left")

In [ ]:
hol = pd.read_csv(
    "/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/holidays_events.csv",
    parse_dates=["date"]
)

hol["is_real_holiday"] = (((hol["transferred"] == False) & (hol["type"] != "Transfer")) | (hol["type"] == "Transfer")).astype(int)
hol_day = hol.groupby("date")["is_real_holiday"].max().reset_index()

all_df = all_df.merge(hol_day, on="date", how="left")

# fill + flags
all_df["is_real_holiday"] = all_df["is_real_holiday"].fillna(0).astype(int)
all_df["holiday_weekday"] = ((all_df["is_real_holiday"]==1) & (all_df["dow"] <= 4)).astype(int)
all_df["holiday_weekend"] = ((all_df["is_real_holiday"]==1) & (all_df["dow"] >= 5)).astype(int)

In [34]:
all_df = all_df.sort_values(["store_nbr","family","date"])
grp = all_df.groupby(["store_nbr","family"], sort=False)

In [28]:
all_df["promo_flag"] = (all_df["onpromotion"] > 0).astype(int)

grp = all_df.groupby(["store_nbr","family"], sort=False)

sales_lags = list(range(1,8)) + [14,21,28,35,42,49,56]
for lag in sales_lags:
    all_df[f"sales_lag_{lag}"] = grp["sales"].shift(lag)

promo_lags = list(range(1,8)) + [14,21,28,35,42,49,56]
for lag in promo_lags:
    all_df[f"promo_lag_{lag}"] = grp["onpromotion"].shift(lag)




In [29]:
bool_cols = all_df.select_dtypes(include=["bool"]).columns
all_df[bool_cols] = all_df[bool_cols].astype(int)

In [30]:
train_fe = all_df[all_df["is_train"]==1].copy()
test_fe  = all_df[all_df["is_train"]==0].copy()

train_fe["y_log"] = np.log1p(train_fe["sales"])

In [31]:
train_model = train_fe.dropna(subset=[f"sales_lag_{max(sales_lags)}"]).copy()

In [32]:
train_model.to_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/train_model.csv", index=False)
test_fe.to_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/test_fe.csv", index=False)